# 07 — CPM & Schedule Analysis
**Wind Farm v4 | Red de precedencias, camino crítico y análisis de holguras**

Este notebook es autónomo: carga `Activity_Master_with_Materials.xlsx` y calcula el CPM a partir de `Predecessors`. Reutiliza el formato de relaciones identificado en el notebook anterior (`24SF;18FS`) y contempla FS, SS, FF y SF.

**Alcance y criterio**
- Se calcula la red completa de actividades (incluidas actividades de duración cero).
- Los cálculos se realizan en días transcurridos, con desfases (`lag`) iguales a cero porque el maestro revisado no contiene un campo de lag explícito.
- Las fechas baseline se conservan para comparación; no se utilizan para forzar artificialmente el resultado CPM.
- El cálculo requiere una red acíclica. Si hay ciclos, se detiene con un mensaje identificable.
- Las relaciones se interpretan como restricciones entre inicios/finales de actividades. No se inventan calendarios laborales.


In [1]:
# https://chatgpt.com/c/6ae46b72-f7d4-83eb-8046-a9c258e2678a

# https://chatgpt.com/c/6aea8156-d1d0-83ed-ba63-1533d7016bd3

## 00. Setup y carga de datos

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display

BASE_DIR = Path(r"C:\\ds4b\\C2011-13 Wind Farm_v4")
FILE_ACTIVITY_MASTER = BASE_DIR / "05_Outputs" / "Activity_Master_with_Materials.xlsx"
OUTPUT_DIR = BASE_DIR / "07_Outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FILE_ACTIVITY_MASTER.exists(), f"No se encuentra el maestro: {FILE_ACTIVITY_MASTER}"

activity_master = pd.read_excel(FILE_ACTIVITY_MASTER)

required = ["ID", "Name", "Predecessors", "Duration_Plan_Days"]
missing = [c for c in required if c not in activity_master.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias en Activity Master: {missing}")

# Normalización mínima
activity_master = activity_master.copy()
activity_master["ID"] = pd.to_numeric(activity_master["ID"], errors="raise").astype(int)
activity_master["Duration_Plan_Days"] = pd.to_numeric(
    activity_master["Duration_Plan_Days"], errors="coerce"
).fillna(0.0).astype(float)

if (activity_master["Duration_Plan_Days"] < 0).any():
    raise ValueError("Hay duraciones negativas en Duration_Plan_Days.")
if activity_master["ID"].duplicated().any():
    dup = activity_master.loc[activity_master["ID"].duplicated(False), "ID"].tolist()
    raise ValueError(f"IDs duplicados en Activity Master: {dup[:20]}")

for col in ["Baseline_Start", "Baseline_End"]:
    if col in activity_master.columns:
        activity_master[col] = pd.to_datetime(activity_master[col], errors="coerce")

activity_master = activity_master.set_index("ID", drop=False).sort_index()

print(f"Actividades cargadas: {len(activity_master)}")
print(f"Duración planificada total (suma simple): {activity_master['Duration_Plan_Days'].sum():,.1f} días")
print("Columnas de fechas baseline disponibles:",
      [c for c in ["Baseline_Start", "Baseline_End"] if c in activity_master.columns])
display(activity_master[["ID", "Name", "Duration_Plan_Days"]].head())


## 01 Construccion de la red CPM

### 1.1 Extraer dependencias desde Predecessors

In [3]:
RELATION_PATTERN = re.compile(
    r"^\s*(\d+)\s*(FS|SS|FF|SF)\s*$",
    re.IGNORECASE
)

relations = []
parse_issues = []

for successor_id, raw_value in activity_master["Predecessors"].items():

    if pd.isna(raw_value) or str(raw_value).strip() == "":
        continue

    # El Excel separa las relaciones mediante punto y coma.
    tokens = str(raw_value).strip().split(";")

    for token in tokens:
        token = token.strip()
        match = RELATION_PATTERN.fullmatch(token)

        if match is None:
            parse_issues.append({
                "Successor_ID": int(successor_id),
                "Token": repr(token),
                "Original_Value": repr(raw_value)
            })
            continue

        predecessor_id = int(match.group(1))
        relation_type = match.group(2).upper()

        relations.append({
            "Predecessor_ID": predecessor_id,
            "Successor_ID": int(successor_id),
            "Relation_Type": relation_type,
            "Lag": 0.0
        })

if parse_issues:
    display(pd.DataFrame(parse_issues))
    raise ValueError(
        "Persisten dependencias sin interpretar. "
        "No se continúa con una red incompleta."
    )

dependencies = pd.DataFrame(relations)

print(f"Relaciones extraídas: {len(dependencies)}")
print("\nTipos de relación:")
display(
    dependencies["Relation_Type"]
    .value_counts()
    .rename_axis("Relation_Type")
    .reset_index(name="Count")
)

### 1.2 Contraste Predecessors / Successors y estructura de red

In [4]:
SUCCESSOR_PATTERN = re.compile(
    r"^\s*(FS|SS|FF|SF)\s*(\d+)\s*$",
    re.IGNORECASE
)

succ_set = set()

pred_set = set(
    dependencies[
        ["Predecessor_ID", "Successor_ID", "Relation_Type"]
    ].itertuples(index=False, name=None)
)

succ_set = set()


if "Successors" in activity_master.columns:

    for predecessor_id, raw_value in activity_master["Successors"].items():

        if pd.isna(raw_value) or str(raw_value).strip() == "":
            continue

        for token in str(raw_value).strip().split(";"):

            token = token.strip()
            match = SUCCESSOR_PATTERN.fullmatch(token)

            if match is None:
                raise ValueError(
                    f"Successor no reconocido en actividad "
                    f"{predecessor_id}: {token!r}"
                )

            relation_type = match.group(1).upper()
            successor_id = int(match.group(2))

            succ_set.add(
                (int(predecessor_id), successor_id, relation_type)
            )

missing_in_successors = pred_set - succ_set
missing_in_predecessors = succ_set - pred_set

print(f"Relaciones en Predecessors: {len(pred_set)}")
print(f"Relaciones en Successors: {len(succ_set)}")
print(f"Solo en Predecessors: {len(missing_in_successors)}")
print(f"Solo en Successors: {len(missing_in_predecessors)}")

if missing_in_successors or missing_in_predecessors:
    display(pd.DataFrame(
        sorted(missing_in_successors),
        columns=["Predecessor_ID", "Successor_ID", "Relation_Type"]
    ))
    raise ValueError("Las relaciones Predecessors/Successors no coinciden.")

print("Contraste correcto: ambas columnas representan las mismas relaciones.")

## 02 Calculo CPM

### 2.1 Preparar restricciones CPM

Todas las relaciones se transforman a una restricción sobre el inicio de la sucesora:

- **FS:** `S_j ≥ S_i + D_i + lag`
- **SS:** `S_j ≥ S_i + lag`
- **FF:** `S_j ≥ S_i + D_i - D_j + lag`
- **SF:** `S_j ≥ S_i - D_j + lag`

Con la red acíclica, el pase hacia delante obtiene los inicios más tempranos. El pase hacia atrás calcula los inicios más tardíos respecto a la duración mínima de la red.


In [5]:
# --------------------------------------------------
# 1. Identificar las hojas WBS
# --------------------------------------------------

df_cpm_base = activity_master.reset_index(drop=True).copy()

wbs_codes = df_cpm_base["WBS"].astype(str).tolist()

df_cpm_base["Has_Children"] = df_cpm_base["WBS"].astype(str).apply(
    lambda parent: any(
        child.startswith(parent + ".")
        for child in wbs_codes
        if child != parent
    )
)

df_cpm_base["Is_WBS_Leaf"] = ~df_cpm_base["Has_Children"]

# Red operativa: solo actividades hoja
df_cpm = (
    df_cpm_base.loc[df_cpm_base["Is_WBS_Leaf"]]
    .copy()
    .reset_index(drop=True)
)

leaf_ids = set(df_cpm["ID"].astype(int))

# --------------------------------------------------
# 2. Comprobar que todas las dependencias
#    pertenecen a las hojas WBS
# --------------------------------------------------

dependency_nodes = (
    set(dependencies["Predecessor_ID"].astype(int))
    | set(dependencies["Successor_ID"].astype(int))
)

outside_nodes = dependency_nodes - leaf_ids

if outside_nodes:
    raise ValueError(
        f"Hay dependencias que involucran nodos no hoja: "
        f"{sorted(outside_nodes)}"
    )

# --------------------------------------------------
# 3. Construir el grafo
# --------------------------------------------------

G = nx.DiGraph()

G.add_nodes_from(leaf_ids)

for dep in dependencies.itertuples(index=False):

    i = int(dep.Predecessor_ID)
    j = int(dep.Successor_ID)

    G.add_edge(i, j)

# --------------------------------------------------
# 4. Validar la red
# --------------------------------------------------

if not nx.is_directed_acyclic_graph(G):
    raise ValueError("La red contiene ciclos. No se puede calcular el CPM.")

topological_order = list(nx.topological_sort(G))

print(f"Actividades hoja WBS: {len(df_cpm)}")
print(f"Nodos del grafo: {G.number_of_nodes()}")
print(f"Aristas del grafo: {G.number_of_edges()}")
print(f"Orden topológico: {len(topological_order)}")

print("Grafo CPM construido correctamente.")

### 2.2 Holgura libre, camino crítico y tabla CPM

In [6]:
# ==================================================
# 1. DURACIONES
# ==================================================

duration = (
    df_cpm.set_index("ID")["Duration_Plan_Days"]
    .astype(float)
    .to_dict()
)

In [7]:
# ==================================================
# 2. TRANSFORMAR RELACIONES EN RESTRICCIONES
#
# S_j >= S_i + weight
# ==================================================

edge_constraints = []

for dep in dependencies.itertuples(index=False):

    i = int(dep.Predecessor_ID)
    j = int(dep.Successor_ID)

    rel = str(dep.Relation_Type).upper().strip()
    lag = float(dep.Lag)

    if rel == "FS":
        weight = duration[i] + lag
    elif rel == "SS":
        weight = lag
    elif rel == "FF":
        weight = duration[i] - duration[j] + lag
    elif rel == "SF":
        weight = -duration[j] + lag
    else:
        raise ValueError(f"Tipo de relación desconocido: {rel}")

    edge_constraints.append((i, j, rel, lag, weight))


print("Restricciones reconstruidas:", len(edge_constraints))

assert len(edge_constraints) == 150, (
    f"Se esperaban 150 restricciones y se han generado {len(edge_constraints)}"
)

In [8]:
# ==================================================
# 3. RESTRICCIONES POR NODO
# ==================================================

pred_constraints = {node: [] for node in G.nodes}
succ_constraints = {node: [] for node in G.nodes}

for i, j, rel, lag, weight in edge_constraints:

    pred_constraints[j].append((i, weight, rel, lag))
    succ_constraints[i].append((j, weight, rel, lag))

In [9]:
# ==================================================
# 4. FORWARD PASS
# ==================================================

ES = {}

for node in topological_order:

    constraints = [
        ES[pred] + weight
        for pred, weight, rel, lag in pred_constraints[node]
    ]

    ES[node] = max([0.0] + constraints)

EF = {
    node: ES[node] + duration[node]
    for node in G.nodes
}

project_duration = max(EF.values()) if EF else 0.0

In [10]:
# ==================================================
# 5. BACKWARD PASS
# Todas las actividades deben finalizar antes del
# fin común del proyecto.
# ==================================================

LS = {}

for node in reversed(topological_order):

    constraints = [
        LS[succ] - weight
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    constraints.append(project_duration - duration[node])

    LS[node] = min(constraints)

LF = {
    node: LS[node] + duration[node]
    for node in G.nodes
}

total_float = {
    node: LS[node] - ES[node]
    for node in G.nodes
}

In [11]:
# ==================================================
# 6. HOLGURA LIBRE
# ==================================================

free_float = {}

for node in G.nodes:

    limits = [
        ES[succ] - (ES[node] + weight)
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    limits.append(project_duration - EF[node])

    free_float[node] = min(limits)

In [12]:
# ==================================================
# 7. TABLA CPM
# ==================================================

cpm = df_cpm[
    [
        "ID",
        "Name",
        "WBS",
        "WBS_Level",
        "Duration_Plan_Days",
        "Baseline_Start",
        "Baseline_End"
    ]
].copy()

cpm["ES_Days"] = cpm["ID"].map(ES)
cpm["EF_Days"] = cpm["ID"].map(EF)
cpm["LS_Days"] = cpm["ID"].map(LS)
cpm["LF_Days"] = cpm["ID"].map(LF)
cpm["Total_Float_Days"] = cpm["ID"].map(total_float)
cpm["Free_Float_Days"] = cpm["ID"].map(free_float)

TOL = 1e-7

cpm["Is_Critical"] = (
    cpm["Total_Float_Days"].abs() <= TOL
)

cpm = (
    cpm.sort_values(["ES_Days", "EF_Days", "ID"])
    .reset_index(drop=True)
)

critical_activities = cpm.loc[cpm["Is_Critical"]].copy()


In [13]:
# ==================================================
# 8. RESUMEN
# ==================================================

print("========== RESULTADOS CPM ==========")
print(f"Actividades: {len(cpm)}")
print(f"Dependencias: {len(edge_constraints)}")
print(f"Duración del proyecto: {project_duration:.2f} días")
print(f"Actividades críticas: {len(critical_activities)}")

print("ES negativos:", int((cpm["ES_Days"] < -TOL).sum()))
print(
    "Holguras totales negativas:",
    int((cpm["Total_Float_Days"] < -TOL).sum())
)

display(cpm.head(20))


In [14]:
"""
# ==================================================
# 1. DURACIONES
# ==================================================

duration = (
    df_cpm.set_index("ID")["Duration_Plan_Days"]
    .astype(float)
    .to_dict()
)

# ==================================================
# 2. TRANSFORMAR RELACIONES EN RESTRICCIONES
#
# S_j >= S_i + weight
# ==================================================

edge_constraints = []

for dep in dependencies.itertuples(index=False):

    i = int(dep.Predecessor_ID)
    j = int(dep.Successor_ID)

    rel = str(dep.Relation_Type).upper().strip()
    lag = float(dep.Lag)

    if rel == "FS":
        weight = duration[i] + lag
    elif rel == "SS":
        weight = lag
    elif rel == "FF":
        weight = duration[i] - duration[j] + lag
    elif rel == "SF":
        weight = -duration[j] + lag
    else:
        raise ValueError(f"Tipo de relación desconocido: {rel}")

    # constraint_slack = ES[j] - (ES[i] + weight)
# ==================================================
# 3. RESTRICCIONES POR NODO
# ==================================================

pred_constraints = {node: [] for node in G.nodes}
succ_constraints = {node: [] for node in G.nodes}

for i, j, rel, lag, weight in edge_constraints:

    pred_constraints[j].append((i, weight, rel, lag))
    succ_constraints[i].append((j, weight, rel, lag))


# ==================================================
# 4. FORWARD PASS
# ==================================================

ES = {}

for node in topological_order:

    constraints = [
        ES[pred] + weight
        for pred, weight, rel, lag in pred_constraints[node]
    ]

    # Ninguna actividad puede comenzar antes del día 0.
    ES[node] = max([0.0] + constraints)


EF = {
    node: ES[node] + duration[node]
    for node in G.nodes
}

# Duración del proyecto: máximo EF de la red operativa.
project_duration = max(EF.values()) if EF else 0.0


# ==================================================
# 5. BACKWARD PASS
# ==================================================

LS = {}


for node in reversed(topological_order):

    # Restricciones impuestas por las sucesoras
    constraints = [
        LS[succ] - weight
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    # Restricción virtual FS(0) hacia el fin del proyecto.
    # Toda actividad debe haber terminado, como máximo,
    # en la fecha de finalización del proyecto.
    constraints.append(
        project_duration - duration[node]
    )

    LS[node] = min(constraints)


# Fechas tardías y holguras
LF = {
    node: LS[node] + duration[node]
    for node in G.nodes
}

total_float = {
    node: LS[node] - ES[node]
    for node in G.nodes
}

# ==================================================
# 6. HOLGURAS
# ==================================================

total_float = {
    node: LS[node] - ES[node]
    for node in G.nodes
}
free_float = {}

for node in G.nodes:

    limits = [
        ES[succ] - (ES[node] + weight)
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    # Restricción virtual de finalización
    limits.append(project_duration - EF[node])

    free_float[node] = min(limits)
    
# ==================================================
# 7. TABLA CPM
# ==================================================

cpm = df_cpm[
    [
        "ID",
        "Name",
        "WBS",
        "WBS_Level",
        "Duration_Plan_Days",
        "Baseline_Start",
        "Baseline_End"
    ]
].copy()

cpm["ES_Days"] = cpm["ID"].map(ES)
cpm["EF_Days"] = cpm["ID"].map(EF)
cpm["LS_Days"] = cpm["ID"].map(LS)
cpm["LF_Days"] = cpm["ID"].map(LF)

cpm["Total_Float_Days"] = cpm["ID"].map(total_float)
cpm["Free_Float_Days"] = cpm["ID"].map(free_float)

cpm["Is_Critical"] = (
    cpm["Total_Float_Days"].abs() <= 1e-7
)

cpm = (
    cpm.sort_values(
        ["ES_Days", "EF_Days", "ID"]
    )
    .reset_index(drop=True)
)

critical_activities = cpm.loc[
    cpm["Is_Critical"]
].copy()


# ==================================================
# 8. RESUMEN
# ==================================================

print("========== RESULTADOS CPM ==========")

print(f"Actividades: {len(cpm)}")
print(f"Dependencias: {len(edge_constraints)}")
print(f"Duración del proyecto: {project_duration:,.2f} días")
print(f"Actividades críticas: {len(critical_activities)}")

print(
    "ES negativos:",
    int((cpm["ES_Days"] < -1e-7).sum())
)

print(
    "Holguras totales negativas:",
    int((cpm["Total_Float_Days"] < -1e-7).sum())
)

display(cpm.head(20))
"""

### 2.3 Resumen del camino crítico y comparación con baseline

In [15]:
# Camino crítico: actividades con holgura total cero ordenadas por inicio temprano.
# Se muestran también las relaciones que conectan actividades críticas.
critical_ids = set(critical_activities["ID"].astype(int))
critical_links = dependencies.loc[
    dependencies["Predecessor_ID"].isin(critical_ids)
    & dependencies["Successor_ID"].isin(critical_ids)
].copy()

critical_path_view = critical_activities[
    ["ID", "Name", "WBS", "Duration_Plan_Days", "ES_Days", "EF_Days",
     "Total_Float_Days", "Free_Float_Days"]
].sort_values(["ES_Days", "ID"])

print("Actividades críticas (ordenadas por ES):")
display(critical_path_view.to_string(index=False))
print("\nRelaciones entre actividades críticas:")
display(critical_links.to_string(index=False))

# Comparación descriptiva entre fechas CPM relativas y fechas baseline.
if {"Baseline_Start", "Baseline_End"}.issubset(cpm.columns):
    baseline_start = activity_master["Baseline_Start"].min()
    if pd.notna(baseline_start):
        cpm["CPM_Start_Date"] = baseline_start + pd.to_timedelta(cpm["ES_Days"], unit="D")
        cpm["CPM_Finish_Date"] = baseline_start + pd.to_timedelta(cpm["EF_Days"], unit="D")
        cpm["Baseline_Start_Variance_Days"] = (
            cpm["CPM_Start_Date"] - cpm["Baseline_Start"]
        ).dt.total_seconds() / 86400
        cpm["Baseline_Finish_Variance_Days"] = (
            cpm["CPM_Finish_Date"] - cpm["Baseline_End"]
        ).dt.total_seconds() / 86400
        display(cpm[[
            "ID", "Name", "Baseline_Start", "CPM_Start_Date",
            "Baseline_End", "CPM_Finish_Date",
            "Baseline_Start_Variance_Days", "Baseline_Finish_Variance_Days"
        ]].head(20))
else:
    print("No hay ambas fechas baseline; se omite la comparación de fechas.")


### 2.4 Impacto de retraso: función reutilizable para escenarios

In [16]:
def recalculate_cpm_with_delays(delay_by_activity):
    """
    Recalcula ES/EF/LS/LF al añadir retrasos a la duración de actividades.

    delay_by_activity: dict {Activity_ID: delay_days}; retraso no negativo.
    Devuelve (tabla_cpm, duración_proyecto).
    """
    adjusted_duration = duration.copy()
    for activity_id, delay in delay_by_activity.items():
        activity_id = int(activity_id)
        if activity_id not in adjusted_duration:
            raise KeyError(f"Activity_ID inexistente: {activity_id}")
        delay = float(delay)
        if delay < 0:
            raise ValueError("Los retrasos deben ser >= 0.")
        adjusted_duration[activity_id] += delay

    es = {node: 0.0 for node in G.nodes}
    for node in topological_order:
        if pred_constraints[node]:
            # Recalcular peso según duración ajustada y tipo de relación
            values = []
            for pred, _, rel, lag in pred_constraints[node]:
                if rel == "FS":
                    w = adjusted_duration[pred] + lag
                elif rel == "SS":
                    w = lag
                elif rel == "FF":
                    w = adjusted_duration[pred] - adjusted_duration[node] + lag
                else:  # SF
                    w = -adjusted_duration[node] + lag
                values.append(es[pred] + w)
            es[node] = max(values)
        else:
            es[node] = 0.0

    ef = {node: es[node] + adjusted_duration[node] for node in G.nodes}
    duration_scenario = max(ef.values()) if ef else 0.0

    ls = {node: np.nan for node in G.nodes}
    for node in reversed(topological_order):
        if succ_constraints[node]:
            limits = []
            for succ, _, rel, lag in succ_constraints[node]:
                if rel == "FS":
                    w = adjusted_duration[node] + lag
                elif rel == "SS":
                    w = lag
                elif rel == "FF":
                    w = adjusted_duration[node] - adjusted_duration[succ] + lag
                else:
                    w = -adjusted_duration[succ] + lag
                limits.append(ls[succ] - w)
            ls[node] = min(limits)
        else:
            ls[node] = duration_scenario - adjusted_duration[node]

    rows = []
    for node in G.nodes:
        rows.append({
            "ID": node,
            "Duration_Scenario_Days": adjusted_duration[node],
            "ES_Days": es[node],
            "EF_Days": ef[node],
            "LS_Days": ls[node],
            "LF_Days": ls[node] + adjusted_duration[node],
            "Total_Float_Days": ls[node] - es[node],
            "Is_Critical": abs(ls[node] - es[node]) <= 1e-7
        })

    return pd.DataFrame(rows).merge(
        activity_master[["ID", "Name"]].reset_index(drop=True),
        on="ID", how="left"
    ), duration_scenario

# Ejemplo de uso (no ejecuta intervención por defecto):
# scenario_cpm, scenario_duration = recalculate_cpm_with_delays({155: 5})
# print("Cambio de duración del proyecto:", scenario_duration - project_duration, "días")


### 2.5 Exportar outputs para el simulador y Power BI

In [17]:
# Tabla CPM principal
cpm.to_csv(OUTPUT_DIR / "cpm_activity_schedule.csv", index=False, encoding="utf-8-sig")
dependencies.to_csv(OUTPUT_DIR / "cpm_dependencies.csv", index=False, encoding="utf-8-sig")
critical_links.to_csv(OUTPUT_DIR / "cpm_critical_links.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([{
    "Project_Duration_CPM_Days": project_duration,
    "Activities": len(cpm),
    "Dependencies": len(dependencies),
    "Critical_Activities": int(cpm["Is_Critical"].sum()),
    "Network_Is_DAG": nx.is_directed_acyclic_graph(G),
    "Lag_Assumption_Days": 0.0
}])
summary.to_csv(OUTPUT_DIR / "cpm_project_summary.csv", index=False, encoding="utf-8-sig")

print("Outputs guardados en:", OUTPUT_DIR)
display(summary)


### 2.6 Lectura de resultados y limitaciones

Antes de conectar el simulador, revisar únicamente:
1. Que la extracción de dependencias coincida con el recuento conocido del proyecto (150 relaciones, si el maestro no ha cambiado).
2. Que la red sea DAG.
3. Que la distribución FS/SS/FF/SF sea coherente con el maestro.
4. Que las actividades críticas y la duración de red tengan sentido frente a la planificación.

**Limitación importante:** los desfases (`lag`) se fijan a cero porque no se ha identificado un campo de desfase en el maestro. Si el Excel codifica desfases dentro del texto de precedencias, habrá que ampliar el parser antes de considerar el CPM definitivo. Asimismo, las fechas baseline con hora (por ejemplo, 17:00) pueden generar diferencias aparentes de fracción de día; el CPM se expresa en días transcurridos y no aplica calendario laboral.

El CPM queda preparado como motor de cálculo y exporta tablas consumibles. La simulación de coste de intervención se desarrollará en el notebook 06; este notebook calcula el impacto en plazo y camino crítico.


In [18]:
print("COLUMNAS DE ACTIVITY_MASTER:")
print(activity_master.columns.tolist())

print("\nÍNDICE:")
print("Nombre:", activity_master.index.name)
print("Tipo:", activity_master.index.dtype)

print("\nPRIMERAS FILAS:")
display(activity_master.head(5))

print("\nTIPOS DE DATOS:")
display(activity_master.dtypes.to_frame("dtype"))

In [19]:
cols = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Predecessors",
    "Successors",
]

display(
    activity_master[cols]
    .sort_index()
    .head(30)
)

print("\nDISTRIBUCIÓN POR NIVEL WBS:")
display(
    activity_master["WBS_Level"]
    .value_counts()
    .sort_index()
    .to_frame("Activities")
)

print("\nACTIVIDADES RAÍZ (IDs 0 y 1):")
display(
    activity_master.loc[[0, 1], cols]
)

In [20]:
# Inspeccionar jerarquía y posibles actividades resumen
df_wbs = activity_master.reset_index(drop=True).copy()

display(
    df_wbs[
        [
            "ID", "Name", "WBS", "WBS_Level",
            "Duration_Plan_Days", "Total_Cost_Plan"
        ]
    ]
    .sort_values(["WBS_Level", "WBS"])
    .head(40)
    .to_string(index=False)
)

# Comprobar cuántos nodos hay en cada nivel
print("\nNODOS POR NIVEL:")
display(
    df_wbs.groupby("WBS_Level").agg(
        Activities=("ID", "count"),
        Duration_Min=("Duration_Plan_Days", "min"),
        Duration_Max=("Duration_Plan_Days", "max"),
        Cost_Total=("Total_Cost_Plan", "sum")
    )
)

In [21]:
df_wbs = activity_master.reset_index(drop=True).copy()

# Un nodo tiene descendientes si existe otro WBS que comienza
# por su código seguido del separador jerárquico "."
wbs_codes = df_wbs["WBS"].astype(str).tolist()

df_wbs["Has_Children"] = df_wbs["WBS"].astype(str).apply(
    lambda parent: any(
        child.startswith(parent + ".")
        for child in wbs_codes
        if child != parent
    )
)

df_wbs["Is_Leaf"] = ~df_wbs["Has_Children"]

print("DISTRIBUCIÓN POR TIPO DE NODO:")
display(
    df_wbs.groupby(["WBS_Level", "Has_Children"]).agg(
        Activities=("ID", "count"),
        Duration_Min=("Duration_Plan_Days", "min"),
        Duration_Max=("Duration_Plan_Days", "max"),
        Cost_Min=("Total_Cost_Plan", "min"),
        Cost_Max=("Total_Cost_Plan", "max")
    )
)

print("\nNODOS RESUMEN:")
display(
    df_wbs.loc[
        df_wbs["Has_Children"],
        ["ID", "Name", "WBS", "WBS_Level",
         "Duration_Plan_Days", "Total_Cost_Plan"]
    ].sort_values("WBS")
)

print("\nNODOS HOJA CON DURACIÓN CERO:")
display(
    df_wbs.loc[
        df_wbs["Is_Leaf"] & df_wbs["Duration_Plan_Days"].eq(0),
        ["ID", "Name", "WBS", "WBS_Level",
         "Duration_Plan_Days", "Total_Cost_Plan"]
    ].sort_values("WBS")
)

In [22]:
# Clasificación WBS frente a dependencias de red
df_check = activity_master.reset_index(drop=True).copy()

# Un nodo WBS hoja no tiene descendientes jerárquicos
wbs_codes = df_check["WBS"].astype(str).tolist()

df_check["Has_Children"] = df_check["WBS"].astype(str).apply(
    lambda parent: any(
        child.startswith(parent + ".")
        for child in wbs_codes
        if child != parent
    )
)

# Interpretar Successors vacíos como ausencia de sucesores
df_check["Has_Successors"] = (
    df_check["Successors"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

df_check["Is_WBS_Leaf"] = ~df_check["Has_Children"]

print("COMPARACIÓN WBS Y RED:")
display(
    pd.crosstab(
        df_check["Is_WBS_Leaf"],
        df_check["Has_Successors"],
        rownames=["WBS hoja"],
        colnames=["Tiene sucesores"]
    )
)

print("\nHOJAS WBS CON SUCESORES:")
display(
    df_check.loc[
        df_check["Is_WBS_Leaf"] & df_check["Has_Successors"],
        ["ID", "Name", "WBS", "Successors"]
    ]
)

print("\nNODOS RESUMEN CON SUCESORES:")
display(
    df_check.loc[
        df_check["Has_Children"] & df_check["Has_Successors"],
        ["ID", "Name", "WBS", "Successors"]
    ]
)

print("\nHOJAS WBS SIN SUCESORES:")
print(
    df_check["Is_WBS_Leaf"].sum()
    - (df_check["Is_WBS_Leaf"] & df_check["Has_Successors"]).sum()
)

In [23]:
# Inspeccionar las dependencias directamente desde activity_master

print("COLUMNAS DISPONIBLES:")
print(activity_master.columns.tolist())

print("\nDEPENDENCIAS DE ACTIVIDADES HOJA:")
display(
    activity_master.loc[
        df_check["Is_WBS_Leaf"].values,
        ["ID", "Name", "WBS", "Predecessors", "Successors"]
    ]
    .head(20)
    .to_string(index=False)
)

print("\nDEPENDENCIAS DE NODOS RESUMEN:")
display(
    activity_master.loc[
        df_check["Has_Children"].values,
        ["ID", "Name", "WBS", "Predecessors", "Successors"]
    ]
    .to_string(index=False)
)

In [24]:
import re
import pandas as pd

df = activity_master.reset_index(drop=True).copy()

leaf_ids = set(
    df.loc[~df_check["Has_Children"].values, "ID"].astype(int)
)

summary_ids = set(
    df.loc[df_check["Has_Children"].values, "ID"].astype(int)
)

pred_pattern = re.compile(r"^\s*(\d+)\s*(FS|SS|FF|SF)\s*$", re.I)
succ_pattern = re.compile(r"^\s*(FS|SS|FF|SF)\s*(\d+)\s*$", re.I)

edges_pred = set()
edges_succ = set()

for _, row in df.iterrows():
    target = int(row["ID"])

    for token in str(row["Predecessors"]).split(";"):
        match = pred_pattern.match(token)
        if match:
            source = int(match.group(1))
            relation = match.group(2).upper()
            edges_pred.add((source, target, relation))

    source = int(row["ID"])

    for token in str(row["Successors"]).split(";"):
        match = succ_pattern.match(token)
        if match:
            relation = match.group(1).upper()
            target = int(match.group(2))
            edges_succ.add((source, target, relation))

print("RELACIONES:")
print("Predecessors:", len(edges_pred))
print("Successors:", len(edges_succ))
print("Diferencias:", len(edges_pred.symmetric_difference(edges_succ)))

print("\nRELACIONES QUE INVOLUCRAN NODOS RESUMEN:")
summary_edges = [
    edge for edge in edges_pred
    if edge[0] in summary_ids or edge[1] in summary_ids
]
print("Total:", len(summary_edges))
print(summary_edges[:20])

print("\nRELACIONES CON EXTREMOS FUERA DE LAS HOJAS:")
invalid_edges = [
    edge for edge in edges_pred
    if edge[0] not in leaf_ids or edge[1] not in leaf_ids
]
print("Total:", len(invalid_edges))
print(invalid_edges[:20])

## 03 Identificacion de las tareas criticas

In [25]:
# ==========================================================
# 07.03 — IDENTIFICACIÓN DE RELACIONES CRÍTICAS
# ==========================================================

critical_edges = []

TOL = 1e-7

for i, j, rel, lag, weight in edge_constraints:

    # Margen entre el inicio real de la sucesora
    # y el inicio mínimo impuesto por la relación.
    constraint_slack = ES[j] - (ES[i] + weight)

    predecessor_critical = abs(total_float[i]) <= TOL
    successor_critical = abs(total_float[j]) <= TOL

    is_binding = abs(constraint_slack) <= TOL

    if is_binding and predecessor_critical and successor_critical:

        critical_edges.append({
            "Predecessor_ID": i,
            "Successor_ID": j,
            "Relation_Type": rel,
            "Lag": lag,
            "Constraint_Weight": weight,
            "Constraint_Slack_Days": constraint_slack
        })

critical_edges_df = pd.DataFrame(critical_edges)

if not critical_edges_df.empty:

    critical_edges_df = critical_edges_df.merge(
        cpm[["ID", "Name"]].rename(
            columns={"ID": "Predecessor_ID", "Name": "Predecessor_Name"}
        ),
        on="Predecessor_ID",
        how="left"
    )

    critical_edges_df = critical_edges_df.merge(
        cpm[["ID", "Name"]].rename(
            columns={"ID": "Successor_ID", "Name": "Successor_Name"}
        ),
        on="Successor_ID",
        how="left"
    )

print("========== RELACIONES CRÍTICAS ==========")
print(f"Relaciones críticas: {len(critical_edges_df)}")

display(critical_edges_df)

In [26]:
# ==========================================================
# DIAGNÓSTICO DE RELACIONES Y ACTIVIDADES CRÍTICAS
# ==========================================================

TOL = 1e-6

# 1. Distribución de holgura total
print("========== HOLGURA TOTAL ==========")

print(
    cpm["Total_Float_Days"]
    .describe(percentiles=[0, 0.05, 0.25, 0.50, 0.75, 0.95, 1])
)

print(
    "\nActividades con holgura aproximadamente cero:",
    int((cpm["Total_Float_Days"].abs() <= TOL).sum())
)

print(
    "Holgura mínima:",
    cpm["Total_Float_Days"].min()
)

# 2. Restricciones vinculantes, sin filtrar por criticidad
edge_diagnostics = []

for i, j, rel, lag, weight in edge_constraints:

    constraint_slack = ES[j] - (ES[i] + weight)

    edge_diagnostics.append({
        "Predecessor_ID": i,
        "Successor_ID": j,
        "Relation_Type": rel,
        "Lag": lag,
        "Weight": weight,
        "Constraint_Slack": constraint_slack,
        "Predecessor_TF": total_float[i],
        "Successor_TF": total_float[j],
        "Is_Binding": abs(constraint_slack) <= TOL
    })

edge_diagnostics_df = pd.DataFrame(edge_diagnostics)

print("\n========== RESTRICCIONES ==========")

print(
    "Relaciones vinculantes:",
    int(edge_diagnostics_df["Is_Binding"].sum())
)

print(
    "Relaciones con holgura negativa:",
    int((edge_diagnostics_df["Constraint_Slack"] < -TOL).sum())
)

print(
    "Distribución de holgura de restricciones:"
)

display(
    edge_diagnostics_df["Constraint_Slack"].describe(
        percentiles=[0, 0.05, 0.25, 0.50, 0.75, 0.95, 1]
    )
)

# 3. Mostrar las actividades con menor holgura
print("\n========== 20 ACTIVIDADES CON MENOR HOLGURA ==========")

display(
    cpm.sort_values("Total_Float_Days")[
        [
            "ID",
            "Name",
            "Duration_Plan_Days",
            "ES_Days",
            "EF_Days",
            "LS_Days",
            "LF_Days",
            "Total_Float_Days"
        ]
    ].head(20)
)

# 4. Mostrar las relaciones más próximas a ser vinculantes
print("\n========== RELACIONES MÁS RESTRICTIVAS ==========")

display(
    edge_diagnostics_df.sort_values("Constraint_Slack")[
        [
            "Predecessor_ID",
            "Successor_ID",
            "Relation_Type",
            "Lag",
            "Constraint_Slack",
            "Predecessor_TF",
            "Successor_TF"
        ]
    ].head(20)
)

In [27]:
print("Aristas de G:", G.number_of_edges())
print("Restricciones edge_constraints:", len(edge_constraints))
print("Filas edge_diagnostics:", len(edge_diagnostics))
print("Columnas edge_diagnostics_df:", edge_diagnostics_df.columns.tolist())

In [28]:
edge_constraints = []

for dep in dependencies.itertuples(index=False):

    i = int(dep.Predecessor_ID)
    j = int(dep.Successor_ID)

    rel = str(dep.Relation_Type).upper().strip()
    lag = float(dep.Lag)

    if rel == "FS":
        weight = duration[i] + lag
    elif rel == "SS":
        weight = lag
    elif rel == "FF":
        weight = duration[i] - duration[j] + lag
    elif rel == "SF":
        weight = -duration[j] + lag
    else:
        raise ValueError(f"Relación desconocida: {rel}")

    edge_constraints.append((i, j, rel, lag, weight))

print("Restricciones reconstruidas:", len(edge_constraints))

In [29]:
# ==========================================================
# DIAGNÓSTICO DE NODOS TERMINALES CPM
# ==========================================================

terminal_nodes = [
    node for node in G.nodes
    if len(list(G.successors(node))) == 0
]

terminal_df = cpm.loc[
    cpm["ID"].isin(terminal_nodes),
    [
        "ID",
        "Name",
        "Duration_Plan_Days",
        "ES_Days",
        "EF_Days",
        "LS_Days",
        "LF_Days",
        "Total_Float_Days"
    ]
].sort_values("EF_Days", ascending=False)

print("========== NODOS TERMINALES ==========")

print("Número de nodos terminales:", len(terminal_nodes))
print("Duración CPM:", project_duration)

display(terminal_df)

print("\n========== NODOS CON MAYOR EF ==========")

display(
    cpm.sort_values(
        "EF_Days",
        ascending=False
    )[
        [
            "ID",
            "Name",
            "ES_Days",
            "EF_Days",
            "LS_Days",
            "LF_Days",
            "Total_Float_Days"
        ]
    ].head(15)
)

In [30]:
"""
# ============================================================
# RELACIONES DE LA CADENA CRÍTICA
# ============================================================

critical_ids = {
    node for node, tf in total_float.items()
    if abs(tf) <= 1e-7
}

critical_relations = []

for u, v, data in G.edges(data=True):

    rel = str(data["Relation_Type"]).upper().strip()
    lag = float(data["Lag"])

    if rel == "FS":
        weight = duration[u] + lag
    elif rel == "SS":
        weight = lag
    elif rel == "FF":
        weight = duration[u] - duration[v] + lag
    elif rel == "SF":
        weight = -duration[v] + lag
    else:
        continue

    constraint_slack = ES[v] - (ES[u] + weight)

    critical_relations.append({
        "Predecessor_ID": u,
        "Successor_ID": v,
        "Relation_Type": rel,
        "Lag": lag,
        "Constraint_Slack": constraint_slack,
        "Predecessor_Float": total_float[u],
        "Successor_Float": total_float[v],
        "Is_Critical_Relation": (
            abs(constraint_slack) <= 1e-7
            and u in critical_ids
            and v in critical_ids
        )
    })

df_critical_relations = pd.DataFrame(critical_relations)

df_critical_relations = (
    df_critical_relations
    .sort_values(
        ["Is_Critical_Relation", "Constraint_Slack"],
        ascending=[False, True]
    )
)

display(
    df_critical_relations[
        (df_critical_relations["Predecessor_ID"].isin(critical_ids))
        | (df_critical_relations["Successor_ID"].isin(critical_ids))
    ]
)
"""

In [31]:
# ============================================================
# RELACIONES DE LA CADENA CRÍTICA
# ============================================================

critical_ids = {
    node for node, tf in total_float.items()
    if abs(tf) <= 1e-7
}

critical_relations = []

for u, v, rel, lag, weight in edge_constraints:

    constraint_slack = ES[v] - (ES[u] + weight)

    critical_relations.append({
        "Predecessor_ID": u,
        "Successor_ID": v,
        "Relation_Type": rel,
        "Lag": lag,
        "Constraint_Slack": constraint_slack,
        "Predecessor_Float": total_float[u],
        "Successor_Float": total_float[v],
        "Is_Critical_Relation": (
            abs(constraint_slack) <= 1e-7
            and u in critical_ids
            and v in critical_ids
        )
    })

df_critical_relations = pd.DataFrame(critical_relations)

df_critical_relations = (
    df_critical_relations
    .sort_values(
        ["Is_Critical_Relation", "Constraint_Slack"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(
    df_critical_relations[
        (df_critical_relations["Predecessor_ID"].isin(critical_ids))
        | (df_critical_relations["Successor_ID"].isin(critical_ids))
    ]
)

In [32]:
print("Nodos del grafo:", G.number_of_nodes())
print("Nodos resumen 0 y 1:", {n for n in [0, 1] if n in G.nodes})

print("\nRelaciones críticas:")
display(
    df_critical_relations[
        df_critical_relations["Is_Critical_Relation"]
    ].to_string(index=False)
)

print("\nActividades críticas:")
display(
    activity_master[
        activity_master["ID"].isin(critical_ids)
    ][["ID", "Name", "Duration_Plan_Days"]]
    .to_string(index=False)
)

In [33]:
print(next(iter(G.edges(data=True))))

In [34]:
print(dependencies.columns.tolist())
display(dependencies.head())

In [35]:
# ============================================================
# INCORPORAR ATRIBUTOS DE LAS RELACIONES AL GRAFO CPM
# ============================================================

for row in dependencies.itertuples(index=False):

    predecessor_id = int(row.Predecessor_ID)
    successor_id = int(row.Successor_ID)

    G.edges[predecessor_id, successor_id].update({
        "Relation_Type": str(row.Relation_Type).upper(),
        "Lag": float(row.Lag)
    })

# Comprobación
print("Número de aristas:", G.number_of_edges())
print("Ejemplo de arista:", next(iter(G.edges(data=True))))

assert G.number_of_edges() == len(dependencies)
assert all(
    "Relation_Type" in data and "Lag" in data
    for _, _, data in G.edges(data=True)
)

print("Atributos incorporados correctamente.")

In [36]:
assert G.number_of_edges() == len(edge_constraints)

for i, j, rel, lag, weight in edge_constraints:
    edge = G.edges[i, j]

    assert edge["Relation_Type"] == rel, (
        f"Tipo distinto en la arista {i} -> {j}"
    )
    assert abs(edge["Lag"] - lag) < 1e-9, (
        f"Lag distinto en la arista {i} -> {j}"
    )

print("OK: las 150 restricciones coinciden con los atributos del grafo.")

In [37]:
# Dependencias directas del hito Finish (ID 166)

print("PREDECESORAS DE FINISH")
display(
    dependencies[dependencies["Successor_ID"] == 166]
)

print("SUCESORAS DE FINISH")
display(
    dependencies[dependencies["Predecessor_ID"] == 166]
)

In [38]:
display(
    activity_master.loc[
        activity_master["ID"].isin([20, 22, 148, 166]),
        [
            "ID", "Name", "WBS", "Duration_Plan_Days",
            "Predecessors", "Successors"
        ]
    ]
)

In [39]:
display(
    dependencies[
        dependencies["Predecessor_ID"].isin([18, 20, 22, 148, 166])
        | dependencies["Successor_ID"].isin([18, 20, 22, 148, 166])
    ].sort_values(
        ["Predecessor_ID", "Successor_ID"],
        ignore_index=True
    )
)

In [40]:
print(dependencies.columns.tolist())

display(
    dependencies.loc[
        dependencies["Predecessor_ID"].isin([18, 20, 22, 140, 148])
        | dependencies["Successor_ID"].isin([18, 20, 22, 140, 148]),
        ["Predecessor_ID", "Successor_ID", "Relation_Type", "Lag"]
    ].to_string(index=False)
)

In [41]:
assert G.number_of_edges() == len(edge_constraints)

for i, j, rel, lag, weight in edge_constraints:
    edge = G.edges[i, j]

    assert edge["Relation_Type"] == rel, (
        f"Tipo distinto en la arista {i} -> {j}"
    )
    assert abs(edge["Lag"] - lag) < 1e-9, (
        f"Lag distinto en la arista {i} -> {j}"
    )

print("OK: las 150 restricciones coinciden con los atributos del grafo.")

In [42]:
# ============================================================
# TABLA FINAL CPM — ACTIVIDADES Y HOLGURAS
# ============================================================

df_cpm_final = pd.DataFrame({
    "ID": list(G.nodes),
    "ES": [ES[n] for n in G.nodes],
    "EF": [EF[n] for n in G.nodes],
    "LS": [LS[n] for n in G.nodes],
    "LF": [LF[n] for n in G.nodes],
    "Total_Float": [total_float[n] for n in G.nodes],
    "Free_Float": [free_float[n] for n in G.nodes],
})

# Evitar ambigüedad entre índice y columna ID
df_activity_info = (
    activity_master[["ID", "Name", "Duration_Plan_Days"]]
    .reset_index(drop=True)
    .copy()
)

df_cpm_final = df_cpm_final.merge(
    df_activity_info,
    on="ID",
    how="left",
    validate="one_to_one"
)

df_cpm_final["Is_Critical"] = (
    df_cpm_final["Total_Float"].abs() <= 1e-7
)

df_cpm_final = (
    df_cpm_final[
        [
            "ID", "Name", "Duration_Plan_Days",
            "ES", "EF", "LS", "LF",
            "Total_Float", "Free_Float", "Is_Critical"
        ]
    ]
    .sort_values(["ES", "ID"])
    .reset_index(drop=True)
)

# ============================================================
# CONTROLES DE INTEGRIDAD
# ============================================================

assert len(df_cpm_final) == 135
assert df_cpm_final["ID"].nunique() == 135
assert df_cpm_final["Name"].notna().all()
assert df_cpm_final["Duration_Plan_Days"].notna().all()
assert (df_cpm_final["Total_Float"] >= -1e-7).all()
assert (df_cpm_final["Free_Float"] >= -1e-7).all()

print("ACTIVIDADES:", len(df_cpm_final))
print("ACTIVIDADES CRÍTICAS:", int(df_cpm_final["Is_Critical"].sum()))
print("DURACIÓN DEL PROYECTO:", project_duration)

display(df_cpm_final)

In [43]:
# ============================================================
# DIAGNÓSTICO DE HITOS Y ACTIVIDADES TERMINALES
# ============================================================

terminal_ids = [
    node for node in G.nodes
    if G.out_degree(node) == 0
]

milestones = df_cpm_final[
    df_cpm_final["Duration_Plan_Days"] == 0
].copy()

print("DURACIÓN GLOBAL CPM:", project_duration)
print("NÚMERO DE ACTIVIDADES TERMINALES:", len(terminal_ids))

print("\nACTIVIDADES TERMINALES:")
display(
    df_cpm_final[
        df_cpm_final["ID"].isin(terminal_ids)
    ][
        ["ID", "Name", "ES", "EF", "Total_Float", "Is_Critical"]
    ].sort_values("EF", ascending=False).to_string(index=False)
)

print("\nHITOS DEL PROYECTO:")
display(
    milestones[
        ["ID", "Name", "ES", "EF", "Total_Float", "Is_Critical"]
    ].sort_values("ES").to_string(index=False)
)

In [44]:
# ============================================================
# CPM GLOBAL — INCORPORACIÓN DE HITO FINAL VIRTUAL
# ============================================================

# 1. Crear una copia del grafo original
G_cpm = G.copy()

virtual_finish_id = max(int(n) for n in G.nodes) + 1

G_cpm.add_node(virtual_finish_id)

duration_cpm = {
    n: float(duration[n])
    for n in G.nodes
}
duration_cpm[virtual_finish_id] = 0.0

# 2. Conservar las 150 dependencias originales
edge_constraints_cpm = list(edge_constraints)

# 3. Vincular todas las actividades al hito final virtual
for node in G.nodes:

    G_cpm.add_edge(
        node,
        virtual_finish_id,
        Relation_Type="FS",
        Lag=0.0
    )

    edge_constraints_cpm.append((
        node,
        virtual_finish_id,
        "FS",
        0.0,
        duration_cpm[node]
    ))

# 4. Reconstruir restricciones de predecesores y sucesores
pred_constraints = {n: [] for n in G_cpm.nodes}
succ_constraints = {n: [] for n in G_cpm.nodes}

for i, j, rel, lag, weight in edge_constraints_cpm:

    pred_constraints[j].append((i, weight, rel, lag))
    succ_constraints[i].append((j, weight, rel, lag))

# 5. Orden topológico
topological_order = list(nx.topological_sort(G_cpm))

# 6. Forward pass
ES = {}

for node in topological_order:

    constraints = [
        ES[pred] + weight
        for pred, weight, rel, lag in pred_constraints[node]
    ]

    ES[node] = max([0.0] + constraints)

EF = {
    node: ES[node] + duration_cpm[node]
    for node in G_cpm.nodes
}

project_duration = EF[virtual_finish_id]

# 7. Backward pass
LS = {virtual_finish_id: project_duration}

for node in reversed(topological_order):

    if node == virtual_finish_id:
        continue

    constraints = [
        LS[succ] - weight
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    if not constraints:
        raise ValueError(
            f"Actividad sin sucesora hacia el hito final: {node}"
        )

    LS[node] = min(constraints)

LF = {
    node: LS[node] + duration_cpm[node]
    for node in G_cpm.nodes
}

# 8. Holguras
total_float = {
    node: LS[node] - ES[node]
    for node in G_cpm.nodes
}

free_float = {}

for node in G_cpm.nodes:

    limits = [
        ES[succ] - (ES[node] + weight)
        for succ, weight, rel, lag in succ_constraints[node]
    ]

    free_float[node] = min(limits) if limits else 0.0

# 9. Validaciones
assert G_cpm.number_of_nodes() == 136
assert len(edge_constraints_cpm) == 285
assert abs(EF[virtual_finish_id] - project_duration) <= 1e-7
assert abs(total_float[virtual_finish_id]) <= 1e-7

assert all(
    total_float[n] >= -1e-7
    for n in G_cpm.nodes
)

assert all(
    free_float[n] >= -1e-7
    for n in G_cpm.nodes
)

# 10. Tabla final de actividades originales
df_activity_info = (
    activity_master[
        ["ID", "Name", "Duration_Plan_Days"]
    ]
    .reset_index(drop=True)
    .copy()
)

df_cpm_final = pd.DataFrame({
    "ID": list(G.nodes),
    "ES": [ES[n] for n in G.nodes],
    "EF": [EF[n] for n in G.nodes],
    "LS": [LS[n] for n in G.nodes],
    "LF": [LF[n] for n in G.nodes],
    "Total_Float": [total_float[n] for n in G.nodes],
    "Free_Float": [free_float[n] for n in G.nodes],
})

df_cpm_final = df_cpm_final.merge(
    df_activity_info,
    on="ID",
    how="left",
    validate="one_to_one"
)

df_cpm_final["Is_Critical"] = (
    df_cpm_final["Total_Float"].abs() <= 1e-7
)

df_cpm_final = (
    df_cpm_final[
        [
            "ID", "Name", "Duration_Plan_Days",
            "ES", "EF", "LS", "LF",
            "Total_Float", "Free_Float", "Is_Critical"
        ]
    ]
    .sort_values(["ES", "ID"])
    .reset_index(drop=True)
)

# 11. Resumen del hito virtual
df_virtual_finish = pd.DataFrame([{
    "ID": virtual_finish_id,
    "Name": "PROJECT FINISH (VIRTUAL)",
    "ES": ES[virtual_finish_id],
    "EF": EF[virtual_finish_id],
    "Duration": 0.0
}])

print("DEPENDENCIAS ORIGINALES:", len(edge_constraints))
print("RESTRICCIONES CPM TOTALES:", len(edge_constraints_cpm))
print("ACTIVIDADES ORIGINALES:", len(df_cpm_final))
print("ACTIVIDADES CRÍTICAS:", int(df_cpm_final["Is_Critical"].sum()))
print("DURACIÓN GLOBAL DEL PROYECTO:", project_duration)

print("\nHITO FINAL VIRTUAL:")
display(df_virtual_finish)

print("\nACTIVIDADES CRÍTICAS:")
display(
    df_cpm_final[df_cpm_final["Is_Critical"]]
)

print("\nTABLA CPM COMPLETA:")
display(df_cpm_final)

In [45]:
# ============================================================
# 07. REGISTRO DE SEGUIMIENTO DE ACTIVIDADES
# ============================================================

# 1. Columnas de información complementaria del maestro
metadata_cols = [
    "ID",
    "WBS",
    "Baseline_Start",
    "Baseline_End",
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost",
    "Material_Type",
    "Is_Critical_Material",
    "Lead_Time_Days",
    "Buffer_Days",
    "Criticality",
    "Material_Readiness_Date",
    "Planned_Anticipation_Days",
    "Supply_Delay_Days",
    "Material_Status",
    "Supply_Risk_Score",
    "Material_Alarm"
]

metadata_cols = [
    c for c in metadata_cols
    if c in activity_master.columns
]

df_metadata = (
    activity_master[metadata_cols]
    .reset_index(drop=True)
    .copy()
)

# 2. Integración CPM + Activity Master
df_tracking = df_cpm_final.copy()

df_tracking = df_tracking.merge(
    df_metadata,
    on="ID",
    how="left",
    validate="one_to_one"
)

# 3. Campos preparados para incorporar ejecución real
df_tracking["Actual_Start"] = pd.NaT
df_tracking["Actual_End"] = pd.NaT
df_tracking["Progress_Pct"] = np.nan
df_tracking["Actual_Cost"] = np.nan
df_tracking["Execution_Status"] = "No informado"

# 4. Orden y estructura
df_tracking = (
    df_tracking
    .sort_values("ID")
    .reset_index(drop=True)
)

# 5. Validaciones
assert len(df_tracking) == 135
assert df_tracking["ID"].is_unique
assert df_tracking["ES"].notna().all()
assert df_tracking["Total_Float"].notna().all()

# 6. Exportación
output_file = OUTPUT_DIR / "Activity_Tracking_Register.xlsx"

df_tracking.to_excel(
    output_file,
    index=False,
    sheet_name="Activity Tracking"
)

print("REGISTRO DE SEGUIMIENTO GENERADO")
print("--------------------------------")
print(f"Actividades: {len(df_tracking)}")
print(f"Actividades críticas: {df_tracking['Is_Critical'].sum()}")
print(f"Duración CPM: {df_tracking['EF'].max():.1f} días")
print(f"Registros de ejecución informados: "
      f"{df_tracking['Actual_Start'].notna().sum()}")
print(f"Archivo: {output_file}")

display(df_tracking.head(10))

In [46]:
# ============================================================
# 08. CONTROL DEL CALENDARIO PLANIFICADO
# ============================================================

df_schedule_control = df_tracking.copy()

# 1. Normalización de fechas y variables
for col in ["Baseline_Start", "Baseline_End"]:
    df_schedule_control[col] = pd.to_datetime(
        df_schedule_control[col], errors="coerce"
    )

for col in ["Supply_Delay_Days", "Buffer_Days", "Supply_Risk_Score"]:
    df_schedule_control[col] = pd.to_numeric(
        df_schedule_control[col], errors="coerce"
    )

# 2. Duración de calendario de la línea base
df_schedule_control["Baseline_Calendar_Days"] = (
    df_schedule_control["Baseline_End"]
    - df_schedule_control["Baseline_Start"]
).dt.total_seconds() / 86400

# 3. Holguras y margen de suministro
df_schedule_control["Supply_Buffer_Excess_Days"] = (
    df_schedule_control["Supply_Delay_Days"]
    - df_schedule_control["Buffer_Days"]
)

# Solo evaluamos el exceso cuando ambos datos están disponibles
valid_supply = (
    df_schedule_control["Supply_Delay_Days"].notna()
    & df_schedule_control["Buffer_Days"].notna()
)

df_schedule_control.loc[
    ~valid_supply, "Supply_Buffer_Excess_Days"
] = np.nan

df_schedule_control["Supply_Buffer_Exceeded"] = (
    valid_supply
    & (df_schedule_control["Supply_Buffer_Excess_Days"] > 0)
)

# 4. Identificación de actividades con riesgo de calendario
# 4. Clasificación del riesgo de calendario

# 4. Clasificación del riesgo de calendario

alarm = (
    df_schedule_control["Material_Alarm"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

df_schedule_control["Supply_Alert"] = (
    ~alarm.isin(["", "OK", "NONE", "NO ALARM"])
)

df_schedule_control["Schedule_Risk"] = (
    df_schedule_control["Supply_Alert"]
    | df_schedule_control["Supply_Buffer_Exceeded"]
)

df_schedule_control["Risk_Level"] = np.select(
    [
        df_schedule_control["Supply_Buffer_Exceeded"],
        df_schedule_control["Supply_Alert"]
    ],
    [
        "Buffer Exceeded",
        "Preventive Alert"
    ],
    default="No Alert"
)

# 5. Orden de prioridad: críticas primero, menor holgura después
df_schedule_control = df_schedule_control.sort_values(
    ["Is_Critical", "Schedule_Risk", "Total_Float"],
    ascending=[False, False, True]
).reset_index(drop=True)

# 6. Resumen de calendario basado en las actividades originales
project_start = df_schedule_control["Baseline_Start"].min()
project_finish = df_schedule_control["Baseline_End"].max()

baseline_calendar_span = (
    project_finish - project_start
).total_seconds() / 86400

project_cpm_duration = df_schedule_control["EF"].max()

print("CONTROL DEL CALENDARIO PLANIFICADO")
print("----------------------------------")
print(f"Actividades analizadas: {len(df_schedule_control)}")
print(f"Inicio de línea base: {project_start}")
print(f"Fin de línea base: {project_finish}")
print(f"Extensión temporal baseline: {baseline_calendar_span:.1f} días naturales")
print(f"Duración CPM: {project_cpm_duration:.1f} días CPM")
print(f"Actividades críticas: {df_schedule_control['Is_Critical'].sum()}")
print(f"Actividades con riesgo de calendario: "
      f"{df_schedule_control['Schedule_Risk'].sum()}")
print(f"Actividades con buffer de suministro excedido: "
      f"{df_schedule_control['Supply_Buffer_Exceeded'].sum()}")

# 7. Exportación
output_file = OUTPUT_DIR / "Schedule_Control.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_schedule_control.to_excel(
        writer, sheet_name="Schedule Control", index=False
    )

print(f"\nArchivo generado: {output_file}")

display(
    df_schedule_control[
        [
            "ID", "Name", "Baseline_Start", "Baseline_End",
            "Baseline_Calendar_Days", "ES", "EF",
            "Total_Float", "Is_Critical",
            "Supply_Delay_Days", "Buffer_Days",
            "Supply_Buffer_Excess_Days", "Schedule_Risk"
        ]
    ].head(15)
)

In [47]:

print("DISTRIBUCIÓN DE MATERIAL_ALARM")
display(
    df_schedule_control["Material_Alarm"]
    .value_counts(dropna=False)
)

print("\nCRUCE ENTRE ALERTAS Y BUFFER")
display(
    pd.crosstab(
        df_schedule_control["Material_Alarm"],
        df_schedule_control["Supply_Buffer_Exceeded"],
        margins=True
    )
)

print("\nACTIVIDADES CON ALERTA PERO SIN EXCESO DE BUFFER")
display(
    df_schedule_control[
        df_schedule_control["Schedule_Risk"]
        & ~df_schedule_control["Supply_Buffer_Exceeded"]
    ][
        ["ID", "Name", "Material_Status", "Material_Alarm",
         "Supply_Delay_Days", "Buffer_Days", "Supply_Risk_Score"]
    ].head(20)
)


In [48]:
print("CLASIFICACIÓN DE RIESGOS")
print("------------------------")

display(
    df_schedule_control["Risk_Level"]
    .value_counts()
    .rename_axis("Risk_Level")
    .to_frame("Activities")
)

print("\nACTIVIDADES CON BUFFER EXCEDIDO:")
display(
    df_schedule_control[
        df_schedule_control["Risk_Level"] == "Buffer Exceeded"
    ][
        ["ID", "Name", "Supply_Delay_Days",
         "Buffer_Days", "Total_Float", "Is_Critical"]
    ]
)

In [49]:
# ============================================================
# 07 — INTEGRATED SCHEDULE EXPOSURE & PRIORITY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Normalización de variables
# ------------------------------------------------------------

df_schedule_control["Total_Float"] = pd.to_numeric(
    df_schedule_control["Total_Float"], errors="coerce"
)

df_schedule_control["Supply_Delay_Days"] = pd.to_numeric(
    df_schedule_control["Supply_Delay_Days"], errors="coerce"
)

df_schedule_control["Buffer_Days"] = pd.to_numeric(
    df_schedule_control["Buffer_Days"], errors="coerce"
)

# Tolerancia numérica para identificar holgura cero
FLOAT_TOLERANCE = 1e-7

# ------------------------------------------------------------
# 2. Identificación de alertas de suministro
# ------------------------------------------------------------

alarm = (
    df_schedule_control["Material_Alarm"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

df_schedule_control["Supply_Alert"] = ~alarm.isin(
    ["", "OK", "NONE", "NO ALARM"]
)

# Recalcular exceso de buffer para mantener consistencia
valid_supply = (
    df_schedule_control["Supply_Delay_Days"].notna()
    & df_schedule_control["Buffer_Days"].notna()
)

df_schedule_control["Supply_Buffer_Excess_Days"] = (
    df_schedule_control["Supply_Delay_Days"]
    - df_schedule_control["Buffer_Days"]
)

df_schedule_control.loc[
    ~valid_supply, "Supply_Buffer_Excess_Days"
] = np.nan

df_schedule_control["Supply_Buffer_Exceeded"] = (
    valid_supply
    & (df_schedule_control["Supply_Buffer_Excess_Days"] > 0)
)

# Existe riesgo de suministro si hay alerta o se supera el buffer
df_schedule_control["Supply_Risk_Flag"] = (
    df_schedule_control["Supply_Alert"]
    | df_schedule_control["Supply_Buffer_Exceeded"]
)

# ------------------------------------------------------------
# 3. Severidad de la incidencia de suministro
# ------------------------------------------------------------

df_schedule_control["Risk_Level"] = np.select(
    [
        df_schedule_control["Supply_Buffer_Exceeded"],
        df_schedule_control["Supply_Alert"]
    ],
    [
        "Buffer Exceeded",
        "Preventive Alert"
    ],
    default="No Alert"
)

# ------------------------------------------------------------
# 4. Exposición de la actividad según su holgura CPM
# ------------------------------------------------------------

critical_exposure = (
    df_schedule_control["Supply_Risk_Flag"]
    & (
        df_schedule_control["Total_Float"].abs()
        <= FLOAT_TOLERANCE
    )
)

df_schedule_control["Schedule_Exposure"] = np.select(
    [
        critical_exposure,
        df_schedule_control["Supply_Risk_Flag"]
    ],
    [
        "Critical activity with supply alert",
        "Supply alert with available float"
    ],
    default="No supply alert"
)

# ------------------------------------------------------------
# 5. Resumen integrado
# ------------------------------------------------------------

print("=" * 65)
print("INTEGRATED SCHEDULE EXPOSURE")
print("=" * 65)

print("\nSUPPLY RISK LEVEL")
print(
    df_schedule_control["Risk_Level"]
    .value_counts()
    .reindex(
        ["No Alert", "Preventive Alert", "Buffer Exceeded"],
        fill_value=0
    )
)

print("\nSCHEDULE EXPOSURE")
print(
    df_schedule_control["Schedule_Exposure"]
    .value_counts()
    .reindex(
        [
            "Critical activity with supply alert",
            "Supply alert with available float",
            "No supply alert"
        ],
        fill_value=0
    )
)

print("\nCROSS-TAB: SUPPLY SEVERITY vs CPM EXPOSURE")
display(
    pd.crosstab(
        df_schedule_control["Risk_Level"],
        df_schedule_control["Schedule_Exposure"],
        margins=True
    )
)

# ------------------------------------------------------------
# 6. Actividades críticas con alertas de suministro
# ------------------------------------------------------------

critical_alerts = (
    df_schedule_control.loc[
        critical_exposure,
        [
            "ID", "Name", "Is_Critical",
            "Total_Float", "Risk_Level",
            "Material_Status", "Supply_Delay_Days",
            "Buffer_Days", "Supply_Buffer_Excess_Days",
            "Material_Alarm"
        ]
    ]
    .sort_values(["Total_Float", "ID"])
)

print("\nCRITICAL ACTIVITIES WITH SUPPLY ALERT")
if critical_alerts.empty:
    print("No critical activities currently have a supply alert.")
else:
    display(critical_alerts)

# ------------------------------------------------------------
# 7. Actividades alertadas con holgura disponible
# ------------------------------------------------------------

activities_with_float = (
    df_schedule_control.loc[
        df_schedule_control["Schedule_Exposure"]
        == "Supply alert with available float",
        [
            "ID", "Name", "Is_Critical",
            "Total_Float", "Risk_Level",
            "Material_Status", "Supply_Delay_Days",
            "Buffer_Days", "Supply_Buffer_Excess_Days",
            "Material_Alarm"
        ]
    ]
    .sort_values(["Total_Float", "Supply_Buffer_Excess_Days"],
                 ascending=[True, False])
)

print("\nSUPPLY ALERTS WITH AVAILABLE CPM FLOAT")
display(activities_with_float.head(20))

# ------------------------------------------------------------
# 8. Exportación actualizada
# ------------------------------------------------------------

df_schedule_control = df_schedule_control.sort_values(
    ["Schedule_Exposure", "Total_Float", "ID"],
    ascending=[True, True, True]
).reset_index(drop=True)

output_file = OUTPUT_DIR / "Schedule_Control.xlsx"

df_schedule_control.to_excel(
    output_file,
    index=False,
    sheet_name="Schedule Control"
)

print("\n" + "=" * 65)
print("SCHEDULE CONTROL EXPORT")
print("=" * 65)
print(f"Activities: {len(df_schedule_control)}")
print(f"Critical activities: {df_schedule_control['Is_Critical'].sum()}")
print(f"Activities with supply risk: {df_schedule_control['Supply_Risk_Flag'].sum()}")
print(f"Critical activities with supply alert: {critical_exposure.sum()}")
print(f"Output: {output_file}")

In [50]:
# ============================================================
# 07 — FREEZE BASELINE FOR SCENARIO SIMULATION
# ============================================================

import copy

df_cpm_baseline = df_cpm_final.copy(deep=True)
df_tracking_baseline = df_tracking.copy(deep=True)
df_schedule_baseline = df_schedule_control.copy(deep=True)

# Baseline summary
baseline_summary = {
    "Activities": len(df_cpm_baseline),
    "Dependencies": 150,
    "Critical_Activities": int(df_cpm_baseline["Is_Critical"].sum()),
    "Project_Duration_Days": float(df_cpm_baseline["EF"].max()),
    "Supply_Risk_Activities": int(
        df_schedule_baseline["Supply_Risk_Flag"].sum()
    ),
    "Buffer_Exceeded": int(
        df_schedule_baseline["Supply_Buffer_Exceeded"].sum()
    )
}

print("BASELINE FROZEN")
display(pd.Series(baseline_summary, name="Baseline"))